## tl;dr
The five compact workloads have **different input-path and indexed-base counts**.
The historical records cover generated suffix mutations, not natural arrivals.
Their old independent-looking insertion check was limited to 64 keys and used an
index-provided snapshot. Do not upgrade those old records to the corrected
oracle merely because the current driver is fixed.

## Context & Methods
This notebook audits existing files, without downloads, WSL, or performance runs.
The paper is the primary artifact; this notebook is its inspectable audit trail.

### Key Assumptions
The recorded `current_size`, `inserted_total`, and `deleted_base_total` fields have
their driver-defined set-cardinality meaning. Infer the original indexed base as
`current_size - inserted_total + deleted_base_total` and require all six rounds
to agree. This is a reconciliation of historical records, not an independent
reproduction of the original source extraction. Depth and byte-length statistics
describe input paths, not the prefix-closed base. Generated regression keys below
are software tests only, never real-corpus performance evidence.


## Data
Record the exact files used by this audit.

In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd
root = Path.cwd()
paths = {
    "legacy_results": root / "results_q1/hrtli_benchmark_results.json",
    "corrected_driver": root / "benchmark_hrtli.py",
    "regression_record": root / "results_q1/reassessment_20260905/compact_oracle_regression.json",
}
source_hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in paths.items()}
pd.DataFrame([{"source": name, "path": str(paths[name].relative_to(root)), "sha256": digest}
              for name, digest in source_hashes.items()])

,source,path,sha256
0,legacy_results,results_q1\hrtli_benchmark_results.json,30a43b778c5ad32dc6afafa6e89add9a2ae0f7a36bf4da...
1,corrected_driver,benchmark_hrtli.py,00fe2c359233db8673fe29355ed94ea563509678c1d4a5...
2,regression_record,results_q1\reassessment_20260905\compact_oracl...,031676f356959c5b98430ba1079cdac5432b73cd032f0e...


## Results
### Reconcile input paths with indexed-base cardinality

In [2]:
legacy = json.loads(paths["legacy_results"].read_text(encoding="utf-8"))
rows = []
for name in ("Filesystem paths", "URL paths", "DNS hierarchy", "JSON paths", "Package paths"):
    result = legacy[name]
    history = result["history"]
    assert [row["round"] for row in history] == list(range(1, 7))
    counts = {row["current_size"] - row["inserted_total"] + row["deleted_base_total"] for row in history}
    assert len(counts) == 1, (name, counts)
    base = counts.pop()
    input_count = result["dataset_stats"]["num_keys"]
    rows.append({"workload": name, "input_paths": input_count, "indexed_base": base,
                 "extra_indexed_keys": base - input_count,
                 "round6_insertions": history[-1]["inserted_total"],
                 "round6_base_deletions": history[-1]["deleted_base_total"],
                 "requested_epsilon": result["base_model"]["target_epsilon"],
                 "realized_epsilon": result["base_model"]["epsilon"]})
profile = pd.DataFrame(rows)
assert (profile["indexed_base"] > profile["input_paths"]).all()
profile

,workload,input_paths,indexed_base,extra_indexed_keys,round6_insertions,round6_base_deletions,requested_epsilon,realized_epsilon
0,Filesystem paths,1011,1028,17,630,13,64,64
1,URL paths,312,502,190,630,13,64,64
2,DNS hierarchy,2600,2880,280,630,13,64,64
3,JSON paths,2600,2602,2,630,13,64,64
4,Package paths,168,704,536,630,13,64,64


### Verify the new guard tests actually ran
A passing current test does not relabel historical data as newly validated.

In [3]:
regression = json.loads(paths["regression_record"].read_text(encoding="utf-8"))
assert regression["exit_code"] == 0
assert "Ran 7 tests" in regression["stderr"] and "OK" in regression["stderr"]
assert regression["sources_sha256"]["benchmark_hrtli.py"] == source_hashes["corrected_driver"]
print(regression["stderr"])
print("All five historical input/base counts differ; the original round-level cardinalities reconcile.")

test_checks_insertions_beyond_old_64_key_sample (test_benchmark_hrtli_audit.BenchmarkAuditTests.test_checks_insertions_beyond_old_64_key_sample) ... ok
test_oracle_does_not_read_index_snapshot (test_benchmark_hrtli_audit.BenchmarkAuditTests.test_oracle_does_not_read_index_snapshot) ... ok
test_rejects_transport_error_even_when_exact_ranks_agree (test_benchmark_hrtli_audit.BenchmarkAuditTests.test_rejects_transport_error_even_when_exact_ranks_agree) ... ok
test_rejects_unrequested_error_instead_of_widening_certificate (test_benchmark_hrtli_audit.BenchmarkAuditTests.test_rejects_unrequested_error_instead_of_widening_certificate) ... ok
test_rejects_wrong_live_size (test_benchmark_hrtli_audit.BenchmarkAuditTests.test_rejects_wrong_live_size) ... ok
test_reports_actual_prefix_closed_base_size (test_benchmark_hrtli_audit.BenchmarkAuditTests.test_reports_actual_prefix_closed_base_size) ... ok
test_update_comparison_constructs_fresh_hrt_state (test_benchmark_hrtli_audit.BenchmarkAuditTests.te

## Takeaways
- **High severity, high confidence:** mislabeled grain understates indexed
  cardinality, especially package paths. Report both counts and identify which
  population the length/depth statistics describe.
- **High severity, high confidence (driver inspection):** generated suffix
  arrivals and prefix closure make these constructed correctness fixtures.
  They are not natural workload timing evidence.
- **High severity, high confidence (driver inspection):** the old rank check
  sampled only 64 insertions from an index snapshot; other exhaustive comparisons
  used the same implementation's exact-rank method. The corrected driver checks
  every live rank against an input-derived set and rejects certificate overflow.
- **Medium severity, high confidence:** the historical update reference began
  HRT-LI after six mutation rounds but built other methods fresh. The driver now
  constructs a fresh HRT-LI state; no old timing is thereby rehabilitated.

No temporal trend is inferred from these single saved artifacts. No new public
dataset extraction was performed, and the original filesystem snapshot has not
been independently reconstructed. Controlled full-scale and specialist-baseline
performance evidence remains pending.
